# Course 03 lab — Vision Transformers

> From image patches to global visual representations

This notebook is self-contained: all teaching code, data generation, model definitions, evaluation, profiling, attention inspection, and artifact creation live here. It uses a deterministic industrial-inspection scenario and official `torchvision` weights.

The lab asks one question repeatedly: **what changed because the image became tokens, and what evidence would justify deploying that choice?**

CPU-safe defaults are deliberately bounded. Set `CV_COURSE_FULL=1` before launching Jupyter for larger generated splits, more training epochs, and more timing repetitions.

![Image-to-token pipeline](assets/patch-token-pipeline.svg)


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import platform
import random
import statistics
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision
from torchvision import transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    ResNet50_Weights,
    Swin_T_Weights,
    ViT_B_16_Weights,
    convnext_tiny,
    resnet50,
    swin_t,
    vit_b_16,
)
from torchvision.models.vision_transformer import interpolate_embeddings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

FULL_RUN = os.getenv("CV_COURSE_FULL", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
COURSE_DIR = Path.cwd()
ARTIFACT_DIR = COURSE_DIR / ".artifacts/vision_transformer_benchmark"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ["Normal", "Micro Scratch", "Alignment Fault", "Contamination"]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(DEVICE),
    "full_run": FULL_RUN,
    "artifact_dir": str(ARTIFACT_DIR),
})


## Phase 1 — Generate a source-aware inspection dataset

The classes require both fine evidence and spatial relationships:

- **Normal** — aligned fasteners and a clean panel;
- **Micro Scratch** — a short, low-width defect that can disappear inside a large patch;
- **Alignment Fault** — one fastener is displaced relative to the others;
- **Contamination** — a local colored deposit with variable area.

Factories A and B supply training/validation examples. Factory C is held out as a source-shift test. A generated dataset makes the mechanism reproducible; it is not production evidence.


In [ ]:
@dataclass
class Sample:
    sample_id: str
    split: str
    source: str
    label: int
    label_name: str
    small_defect: bool
    image: Image.Image


def make_panel(label: int, source: str, seed: int, size: int = 224) -> tuple[Image.Image, bool]:
    rng = np.random.default_rng(seed)
    base = {"Factory-A": (194, 201, 207), "Factory-B": (182, 196, 207), "Factory-C": (205, 190, 172)}[source]
    array = np.zeros((size, size, 3), dtype=np.float32)
    array[...] = np.array(base, dtype=np.float32)
    gradient = np.linspace(-12, 12, size, dtype=np.float32)[None, :, None]
    if source == "Factory-B":
        gradient = gradient.transpose(1, 0, 2)
    if source == "Factory-C":
        gradient = -gradient
    array += gradient + rng.normal(0, 3.0 if source != "Factory-C" else 5.0, array.shape)
    image = Image.fromarray(np.uint8(np.clip(array, 0, 255)))
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((24, 24, 200, 200), radius=16, outline=(66, 76, 84), width=4, fill=None)
    anchors = [(68, 68), (156, 68), (68, 156), (156, 156)]
    if label == 2:
        anchors[-1] = (174, 142)
    for x, y in anchors:
        draw.ellipse((x - 14, y - 14, x + 14, y + 14), fill=(94, 101, 107), outline=(40, 45, 50), width=3)
        draw.line((x - 7, y, x + 7, y), fill=(205, 211, 214), width=3)

    small = False
    if label == 1:
        x, y = int(rng.integers(85, 135)), int(rng.integers(94, 130))
        length = int(rng.integers(10, 17))
        draw.line((x, y, x + length, y + int(rng.integers(-3, 4))), fill=(72, 42, 38), width=2)
        small = True
    elif label == 3:
        radius = int(rng.choice([5, 6, 12, 16]))
        x, y = int(rng.integers(75, 150)), int(rng.integers(80, 145))
        draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=(118, 84, 45), outline=(82, 55, 29), width=2)
        small = radius <= 6

    if source == "Factory-C":
        image = ImageEnhance.Contrast(image).enhance(0.86).filter(ImageFilter.GaussianBlur(0.35))
    return image, small


def build_samples() -> list[Sample]:
    counts = {"train": 10 if FULL_RUN else 4, "val": 4 if FULL_RUN else 2, "test": 8 if FULL_RUN else 3}
    records: list[Sample] = []
    for source_index, source in enumerate(["Factory-A", "Factory-B"]):
        for split in ["train", "val"]:
            for label, label_name in enumerate(CLASS_NAMES):
                for index in range(counts[split]):
                    seed = 10_000 * source_index + 1_000 * label + 100 * (split == "val") + index
                    image, small = make_panel(label, source, seed)
                    records.append(Sample(f"{source}-{split}-{label}-{index}", split, source, label, label_name, small, image))
    for label, label_name in enumerate(CLASS_NAMES):
        for index in range(counts["test"]):
            seed = 90_000 + 1_000 * label + index
            image, small = make_panel(label, "Factory-C", seed)
            records.append(Sample(f"Factory-C-test-{label}-{index}", "test", "Factory-C", label, label_name, small, image))
    return records


samples = build_samples()
sample_frame = pd.DataFrame([{k: v for k, v in asdict(item).items() if k != "image"} for item in samples])
display(sample_frame.groupby(["split", "source", "label_name"]).size().rename("count").reset_index())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for column, label_name in enumerate(CLASS_NAMES):
    for row, source in enumerate(["Factory-A", "Factory-C"]):
        item = next(s for s in samples if s.label_name == label_name and s.source == source)
        axes[row, column].imshow(item.image)
        axes[row, column].set_title(f"{source}\n{label_name}")
        axes[row, column].axis("off")
plt.tight_layout()


## Phase 2 — Patchification and the information bottleneck

For square inputs and patches, $N=(H/P)(W/P)$. The visualization makes the first irreversible design choice visible: a token summarizes everything inside one patch before attention begins.


In [ ]:
def patchify(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    assert images.ndim == 4
    b, c, h, w = images.shape
    assert h % patch_size == 0 and w % patch_size == 0
    patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    return patches.permute(0, 2, 3, 1, 4, 5).reshape(b, -1, c * patch_size * patch_size)


example = samples[1].image
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, patch_size in zip(axes, [32, 16, 8]):
    axis.imshow(example)
    for value in range(0, example.width + 1, patch_size):
        axis.axvline(value, color="#F59E42", linewidth=0.7)
        axis.axhline(value, color="#F59E42", linewidth=0.7)
    token_count = (example.width // patch_size) ** 2
    axis.set_title(f"P={patch_size}: {token_count} tokens\n{token_count**2:,} score entries/head")
    axis.axis("off")
plt.tight_layout()

image_tensor = transforms.ToTensor()(example).unsqueeze(0)
for patch_size in [32, 16, 8]:
    result = patchify(image_tensor, patch_size)
    print(f"patch={patch_size:2d} -> shape={tuple(result.shape)}")


### Linear patch projection equals a strided convolution

`Conv2d(C, D, kernel_size=P, stride=P)` applies the same learned projection at every non-overlapping patch. We copy the linear weights into the convolution kernel and require numerical equality.


In [ ]:
torch.manual_seed(SEED)
batch = torch.randn(2, 3, 32, 32)
patch_size, embed_dim = 8, 12
linear = nn.Linear(3 * patch_size * patch_size, embed_dim)
convolution = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
with torch.no_grad():
    convolution.weight.copy_(linear.weight.reshape(embed_dim, 3, patch_size, patch_size))
    convolution.bias.copy_(linear.bias)

linear_tokens = linear(patchify(batch, patch_size))
conv_tokens = convolution(batch).flatten(2).transpose(1, 2)
max_error = (linear_tokens - conv_tokens).abs().max().item()
assert torch.allclose(linear_tokens, conv_tokens, atol=1e-6)
print({"linear_shape": tuple(linear_tokens.shape), "conv_shape": tuple(conv_tokens.shape), "max_abs_error": max_error})


## Phase 3 — Build scaled dot-product attention by hand

![Query-key-value attention](assets/attention-qkv.svg)

The example keeps only four tokens and one head. It prints each intermediate tensor, checks that every attention row sums to one, and verifies the output with PyTorch's common scaled-dot-product-attention primitive.


In [ ]:
tokens = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
])
w_q = torch.tensor([[1., 0.], [0., 1.], [1., 0.], [0., 1.]])
w_k = torch.tensor([[1., 0.], [0., 1.], [0., 1.], [1., 0.]])
w_v = torch.tensor([[1., 0.], [0., 1.], [0.5, 0.5], [1., -1.]])
q, k, v = tokens @ w_q, tokens @ w_k, tokens @ w_v
logits = q @ k.T / math.sqrt(q.shape[-1])
weights = logits.softmax(dim=-1)
manual_output = weights @ v
pytorch_output = F.scaled_dot_product_attention(
    q[None, None], k[None, None], v[None, None], dropout_p=0.0
).squeeze(0).squeeze(0)

assert torch.allclose(weights.sum(dim=-1), torch.ones(4))
assert torch.allclose(manual_output, pytorch_output, atol=1e-6)
for name, tensor in {"Q": q, "K": k, "V": v, "scaled logits": logits, "attention weights": weights, "output": manual_output}.items():
    print(f"\n{name} {tuple(tensor.shape)}\n{tensor.round(decimals=3)}")


### Attention memory: count values before quoting gigabytes

Dense attention semantics involve an $N\times N$ matrix per head. A naive implementation that explicitly retains those weights requires $B\,h\,N^2$ values per layer. The calculation below makes the growth concrete for ViT-B/16 at 224, 384, and 768 pixels.

This is **attention-weight storage only**, not total model memory. Q/K/V tensors, outputs, MLP activations, parameters, gradients, optimizer state, allocator behavior, kernel workspaces, batch size, and precision also matter. Fused kernels may avoid materializing the complete score/probability matrix in high-bandwidth memory.

### Same semantics, different execution

The mathematical expression does not require a naive sequence of separately materialized operations:

```text
QKᵀ → store scores → softmax → store weights → multiply V

versus

tile Q/K/V → update softmax statistics → accumulate output → write result
```

`torch.nn.functional.scaled_dot_product_attention` provides one semantic API and dispatches to an available math, memory-efficient, or FlashAttention-style implementation when its device, dtype, shapes, mask, and build support permit. FlashAttention is IO-aware: it reorganizes exact dense attention to reduce movement between slower high-bandwidth memory and faster on-chip memory. It does not make the pairwise arithmetic of global attention linear, and it is not guaranteed to be faster for every shape or backend.


In [ ]:
def attention_weight_memory(
    resolution: int,
    patch_size: int = 16,
    heads: int = 12,
    layers: int = 12,
    bytes_per_value: int = 4,
) -> dict:
    assert resolution % patch_size == 0
    patches = (resolution // patch_size) ** 2
    tokens = patches + 1
    values = heads * tokens * tokens
    mib_per_layer = values * bytes_per_value / 2**20
    return {
        "resolution": resolution,
        "patches": patches,
        "tokens_with_cls": tokens,
        "heads": heads,
        "precision": "FP32" if bytes_per_value == 4 else "FP16/BF16",
        "attention_values_per_layer": values,
        "attention_weight_mib_per_layer_sample": mib_per_layer,
        "naive_weight_mib_all_layers_sample": layers * mib_per_layer,
    }


attention_memory = pd.DataFrame([
    attention_weight_memory(resolution, bytes_per_value=bytes_per_value)
    for resolution in [224, 384, 768]
    for bytes_per_value in [4, 2]
])
display(attention_memory.round(3))

vit_224 = attention_memory.query("resolution == 224 and precision == 'FP32'").iloc[0]
assert int(vit_224.attention_values_per_layer) == 12 * 197 * 197
print(
    f"ViT-B/16 at 224: 12 × 197 × 197 = {int(vit_224.attention_values_per_layer):,} values; "
    f"FP32 ≈ {vit_224.attention_weight_mib_per_layer_sample:.2f} MiB per layer per sample."
)


def global_vs_window_interactions(token_grid_side: int, window_side: int) -> dict:
    assert token_grid_side % window_side == 0
    tokens = token_grid_side**2
    windows = (token_grid_side // window_side) ** 2
    tokens_per_window = window_side**2
    global_scores = tokens**2
    window_scores = windows * tokens_per_window**2
    assert window_scores == tokens * tokens_per_window
    return {
        "token_grid": f"{token_grid_side}×{token_grid_side}",
        "tokens": tokens,
        "window": f"{window_side}×{window_side}",
        "global_interactions_per_head": global_scores,
        "window_interactions_per_head": window_scores,
        "global_to_window_ratio": global_scores / window_scores,
    }


attention_cost = pd.DataFrame([global_vs_window_interactions(56, 7)])
display(attention_cost)
assert int(attention_cost.iloc[0].global_interactions_per_head) == 9_834_496
assert int(attention_cost.iloc[0].window_interactions_per_head) == 153_664

sdpa_environment = {
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "flash_attention_compiled": (
        torch.backends.cuda.is_flash_attention_available()
        if hasattr(torch.backends.cuda, "is_flash_attention_available") else False
    ),
    "semantic_api": "torch.nn.functional.scaled_dot_product_attention",
    "note": "Actual backend dispatch depends on device, dtype, shape, masks, dropout, and build support.",
}
print(sdpa_environment)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fp32_memory = attention_memory.query("precision == 'FP32'")
axes[0].plot(fp32_memory.resolution, fp32_memory.attention_weight_mib_per_layer_sample, marker="o")
axes[0].set_yscale("log")
axes[0].set(xlabel="Resolution", ylabel="FP32 attention-weight MiB / layer / sample", title="Dense attention storage grows rapidly")
cost_values = attention_cost.iloc[0]
axes[1].bar(
    ["Global", "7×7 windows"],
    [cost_values.global_interactions_per_head, cost_values.window_interactions_per_head],
    color=["#7667E8", "#F59E42"],
)
axes[1].set_yscale("log")
axes[1].set(ylabel="Pairwise scores per head (log scale)", title="56×56 tokens: global vs window attention")
plt.tight_layout()


## Phase 4 — Implement a minimal pre-norm Vision Transformer

![Pre-normalized transformer block](assets/transformer-encoder-block.svg)

The implementation exposes all important contracts. It is intentionally educational: production libraries add optimized kernels, stochastic depth, richer initialization, compilation paths, and checkpoint compatibility.


In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, image_size: int, patch_size: int, channels: int, embed_dim: int):
        super().__init__()
        assert image_size % patch_size == 0
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.projection = nn.Conv2d(channels, embed_dim, patch_size, patch_size)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        tokens = self.projection(images).flatten(2).transpose(1, 2)
        assert tokens.shape[1] == self.num_patches
        return tokens


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, heads: int):
        super().__init__()
        assert embed_dim % heads == 0
        self.heads = heads
        self.head_dim = embed_dim // heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)
        self.last_attention: torch.Tensor | None = None

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        b, n, d = tokens.shape
        qkv = self.qkv(tokens).reshape(b, n, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = scores.softmax(dim=-1)
        self.last_attention = attention.detach()
        mixed = (attention @ v).transpose(1, 2).reshape(b, n, d)
        return self.output(mixed)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, heads: int, mlp_ratio: int = 4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadSelfAttention(embed_dim, heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_ratio * embed_dim), nn.GELU(), nn.Linear(mlp_ratio * embed_dim, embed_dim)
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        tokens = tokens + self.attention(self.norm1(tokens))
        return tokens + self.mlp(self.norm2(tokens))


class TinyViT(nn.Module):
    def __init__(self, image_size=64, patch_size=16, num_classes=4, embed_dim=48, depth=2, heads=4, pooling="mean"):
        super().__init__()
        self.patch_embed = PatchEmbed(image_size, patch_size, 3, embed_dim)
        self.class_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.position = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self.pooling = pooling
        nn.init.trunc_normal_(self.position, std=0.02)
        nn.init.trunc_normal_(self.class_token, std=0.02)

    def forward_features(self, images: torch.Tensor) -> torch.Tensor:
        patches = self.patch_embed(images)
        cls = self.class_token.expand(images.shape[0], -1, -1)
        tokens = torch.cat([cls, patches], dim=1) + self.position
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)
        return tokens[:, 0] if self.pooling == "cls" else tokens[:, 1:].mean(dim=1)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.head(self.forward_features(images))


tiny = TinyViT()
test_output = tiny(torch.randn(2, 3, 64, 64))
assert test_output.shape == (2, len(CLASS_NAMES))
assert tiny.blocks[0].attention.last_attention.shape == (2, 4, 17, 17)
print(tiny)
print("shape contract:", tuple(test_output.shape), "attention:", tuple(tiny.blocks[0].attention.last_attention.shape))


## Phase 5 — Patch-size experiment

The same bounded training loop compares patches 32, 16, and 8. Because this is a small generated dataset and a tiny network, the quality values are teaching observations—not claims about ViT scaling.


In [ ]:
def tensor_for_sample(sample: Sample, size: int) -> torch.Tensor:
    return transforms.ToTensor()(sample.image.resize((size, size), Image.Resampling.BILINEAR))


train_like = [s for s in samples if s.split in {"train", "val"}]
test_like = [s for s in samples if s.split == "test"]
x_train = torch.stack([tensor_for_sample(s, 64) for s in train_like])
y_train = torch.tensor([s.label for s in train_like])
x_test = torch.stack([tensor_for_sample(s, 64) for s in test_like])
y_test = np.array([s.label for s in test_like])


def latency_ms(model: nn.Module, shape: tuple[int, ...], repeats: int = 6) -> list[float]:
    model.eval()
    device = next(model.parameters()).device
    value = torch.randn(*shape, device=device)
    with torch.inference_mode():
        for _ in range(2):
            model(value)
        if device.type == "cuda":
            torch.cuda.synchronize()
        timings = []
        for _ in range(repeats):
            start = time.perf_counter()
            model(value)
            if device.type == "cuda":
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - start) * 1000)
    return timings


def train_patch_variant(patch_size: int) -> dict:
    torch.manual_seed(SEED + patch_size)
    model = TinyViT(patch_size=patch_size, embed_dim=48, depth=2, heads=4, pooling="mean").to(DEVICE)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(SEED))
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
    start = time.perf_counter()
    model.train()
    for _ in range(6 if FULL_RUN else 3):
        for images, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(images.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
    train_seconds = time.perf_counter() - start
    model.eval()
    with torch.inference_mode():
        predictions = model(x_test.to(DEVICE)).argmax(dim=1).cpu().numpy()
    timings = latency_ms(model, (1, 3, 64, 64), 12 if FULL_RUN else 5)
    tokens = (64 // patch_size) ** 2 + 1
    interaction_bytes = 2 * 4 * tokens * tokens * 4
    return {
        "patch_size": patch_size,
        "tokens_with_cls": tokens,
        "score_entries_all_heads_layers": 2 * 4 * tokens * tokens,
        "attention_estimate_mb_fp32": interaction_bytes / 2**20,
        "macro_f1": f1_score(y_test, predictions, average="macro", zero_division=0),
        "small_defect_recall": recall_score(
            y_test[[s.small_defect for s in test_like]], predictions[[s.small_defect for s in test_like]], average="macro", zero_division=0
        ) if any(s.small_defect for s in test_like) else float("nan"),
        "train_seconds": train_seconds,
        "median_ms_b1": float(np.median(timings)),
    }


patch_results = pd.DataFrame([train_patch_variant(patch) for patch in [32, 16, 8]])
display(patch_results.round(4))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(patch_results.patch_size, patch_results.macro_f1, marker="o")
axes[0].invert_xaxis(); axes[0].set(xlabel="Patch size", ylabel="Macro F1", title="Quality is measured, not assumed")
axes[1].plot(patch_results.tokens_with_cls, patch_results.median_ms_b1, marker="o", color="#F59E42")
axes[1].set(xlabel="Tokens including CLS", ylabel="Median latency (ms)", title="Token budget changes runtime")
plt.tight_layout()


## Phase 6 — Controlled official-backbone comparison

![CNN, ViT, and Swin comparison](assets/cnn-vit-swin-comparison.svg)

All four encoders use official weights, the same 224-pixel tensor and ImageNet normalization contract, frozen weights, the same source-aware data, and the same standardized logistic-regression probe. Pretraining recipes and model capacities still differ, so this is a selection benchmark—not a causal architecture experiment.


In [ ]:
COMMON_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ImageDataset(Dataset):
    def __init__(self, rows: list[Sample], transform):
        self.rows, self.transform = rows, transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        item = self.rows[index]
        return self.transform(item.image), item.label


def build_encoder(name: str, image_size: int = 224, state: dict | None = None) -> nn.Module:
    if name == "ResNet-50":
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Identity()
    elif name == "ConvNeXt-Tiny":
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        model.classifier[-1] = nn.Identity()
    elif name == "ViT-B/16":
        if state is None and image_size == 224:
            model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
            model.heads = nn.Identity()
        else:
            model = vit_b_16(weights=None, image_size=image_size)
            model.heads = nn.Identity()
            adapted = interpolate_embeddings(image_size, 16, copy.deepcopy(state), reset_heads=True)
            model.load_state_dict(adapted, strict=True)
    elif name == "Swin-T":
        model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
        model.head = nn.Identity()
    else:
        raise KeyError(name)
    return model.eval().to(DEVICE)


def extract_features(model: nn.Module, rows: list[Sample], transform, batch_size: int = 4) -> np.ndarray:
    loader = DataLoader(ImageDataset(rows, transform), batch_size=batch_size, shuffle=False)
    features = []
    with torch.inference_mode():
        for images, _ in loader:
            features.append(model(images.to(DEVICE)).reshape(len(images), -1).cpu())
    return torch.cat(features).numpy()


def probe_metrics(rows: list[Sample], predictions: np.ndarray) -> dict:
    truth = np.array([row.label for row in rows])
    defects = truth != 0
    small = np.array([row.small_defect for row in rows])
    return {
        "macro_f1": f1_score(truth, predictions, average="macro", zero_division=0),
        "defect_recall": recall_score(defects, predictions != 0, zero_division=0),
        "small_defect_recall": recall_score(truth[small], predictions[small], average="macro", zero_division=0) if small.any() else float("nan"),
        **{f"recall_{CLASS_NAMES[i].lower().replace(' ', '_')}": value for i, value in enumerate(recall_score(truth, predictions, average=None, labels=range(len(CLASS_NAMES)), zero_division=0))},
    }


def profile_encoder(model: nn.Module, resolution: int = 224) -> dict:
    repeats = 10 if FULL_RUN else 4
    batch1 = latency_ms(model, (1, 3, resolution, resolution), repeats)
    batch4 = latency_ms(model, (4, 3, resolution, resolution), max(3, repeats // 2))
    array = np.array(batch1)
    return {
        "median_ms_b1": float(np.median(array)),
        "p90_ms_b1": float(np.percentile(array, 90)),
        "p95_ms_b1": float(np.percentile(array, 95)),
        "iqr_ms_b1": float(np.percentile(array, 75) - np.percentile(array, 25)),
        "throughput_images_s_b4": float(4_000 / np.median(batch4)),
    }


train_rows = [row for row in samples if row.split in {"train", "val"}]
test_rows = [row for row in samples if row.split == "test"]
quality_records, system_records, confusion_records = [], [], {}
vit_encoder = vit_probe = vit_test_features = vit_test_predictions = vit_test_probabilities = None

for model_name in ["ResNet-50", "ConvNeXt-Tiny", "ViT-B/16", "Swin-T"]:
    print(f"\nLoading and evaluating {model_name} ...")
    model = build_encoder(model_name)
    train_features = extract_features(model, train_rows, COMMON_TRANSFORM)
    test_features = extract_features(model, test_rows, COMMON_TRANSFORM)
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0, random_state=SEED))
    probe.fit(train_features, [row.label for row in train_rows])
    predictions = probe.predict(test_features)
    probabilities = probe.predict_proba(test_features)
    quality_records.append({"model": model_name, **probe_metrics(test_rows, predictions)})
    confusion_records[model_name] = confusion_matrix([row.label for row in test_rows], predictions).tolist()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    record = {
        "model": model_name,
        "parameters_m": parameter_count / 1e6,
        "state_size_mb_fp32": sum(p.numel() * p.element_size() for p in model.parameters()) / 2**20,
        "resolution": 224,
        "patch_tokens": 197 if model_name == "ViT-B/16" else 3136 if model_name == "Swin-T" else None,
        "attention_interactions_estimate": 12 * 12 * 197**2 if model_name == "ViT-B/16" else None,
        **profile_encoder(model),
    }
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.inference_mode():
            model(torch.randn(1, 3, 224, 224, device=DEVICE))
        record["peak_accelerator_memory_mb"] = torch.cuda.max_memory_allocated() / 2**20
    else:
        record["peak_accelerator_memory_mb"] = None
    system_records.append(record)
    if model_name == "ViT-B/16":
        vit_encoder, vit_probe = model.to("cpu"), probe
        vit_test_features, vit_test_predictions, vit_test_probabilities = test_features, predictions, probabilities
    else:
        del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

quality_results = pd.DataFrame(quality_records)
systems_results = pd.DataFrame(system_records)
display(quality_results.round(4))
display(systems_results.round(3))


### Read the quality and systems tables together

Macro F1 weights all four classes equally. Defect recall treats every non-normal sample as requiring intervention. Small-defect recall isolates the examples most likely to be damaged by coarse tokenization or resizing. All test images come from held-out Factory C, so the clean test table is also the source-shift result.

Median, p90, p95, and IQR summarize repeated measurements. These are **demonstration measurements for this notebook runtime only**. They are not target-hardware contracts.


In [ ]:
merged_results = quality_results.merge(systems_results, on="model")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(merged_results.model, merged_results.macro_f1, color=["#2878C8", "#16A3A5", "#7667E8", "#F59E42"])
axes[0].set(ylabel="Held-out-source macro F1", ylim=(0, 1.05), title="Representation quality")
axes[0].tick_params(axis="x", rotation=25)
axes[1].errorbar(
    merged_results.median_ms_b1, merged_results.macro_f1,
    xerr=merged_results.iqr_ms_b1 / 2, fmt="o", color="#16324F", ecolor="#8492A6", capsize=4
)
for _, row in merged_results.iterrows():
    axes[1].annotate(row.model, (row.median_ms_b1, row.macro_f1), xytext=(5, 5), textcoords="offset points")
axes[1].set(xlabel="Batch-1 median latency (ms)", ylabel="Macro F1", title="Quality–latency evidence")
plt.tight_layout()


## Phase 7 — Resolution shift with position interpolation

ViT-B/16 was pretrained at 224 pixels. For lower and higher inputs, the learned patch-position grid is reshaped, bicubically interpolated, and loaded into a model with the new image-size contract. The class-token position is preserved.

Interpolation repairs a shape mismatch; it does not guarantee stable quality. The probe and runtime are remeasured at every resolution.


In [ ]:
vit_encoder = vit_encoder.to("cpu")
base_vit_state = copy.deepcopy(vit_encoder.state_dict())


def transform_at(resolution: int):
    return transforms.Compose([
        transforms.Resize((resolution, resolution), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


resolution_records = []
for resolution in [160, 224, 288]:
    if resolution == 224:
        model = vit_encoder.to(DEVICE)
    else:
        model = build_encoder("ViT-B/16", image_size=resolution, state=base_vit_state)
    transform = transform_at(resolution)
    train_features = extract_features(model, train_rows, transform)
    test_features = extract_features(model, test_rows, transform)
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=SEED))
    probe.fit(train_features, [row.label for row in train_rows])
    predictions = probe.predict(test_features)
    resolution_records.append({
        "resolution": resolution,
        "patch_tokens_with_cls": (resolution // 16) ** 2 + 1,
        **probe_metrics(test_rows, predictions),
        **profile_encoder(model, resolution),
    })
    if resolution == 224:
        vit_encoder = model.to("cpu")
    else:
        del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

resolution_results = pd.DataFrame(resolution_records)
display(resolution_results[["resolution", "patch_tokens_with_cls", "macro_f1", "small_defect_recall", "median_ms_b1", "p95_ms_b1"]].round(4))

fig, axis = plt.subplots(figsize=(7, 4))
axis.plot(resolution_results.resolution, resolution_results.macro_f1, marker="o", label="Macro F1")
axis.plot(resolution_results.resolution, resolution_results.small_defect_recall, marker="s", label="Small-defect recall")
axis.set(xlabel="Input resolution", ylabel="Score", ylim=(0, 1.05), title="Interpolation enables evaluation—not invariance")
axis.legend(); plt.tight_layout()


### See position interpolation as a two-dimensional operation

The learned ViT-B/16 patch-position table contains a $14\times14$ spatial grid at 224 pixels. To support 384 pixels, the grid becomes $24\times24$. The notebook selects three embedding dimensions with visible spatial variation and plots the original and bicubically interpolated grids using a shared color scale.

The visualization explains the tensor operation only. The $24\times24$ values were interpolated, not learned from new-resolution evidence, so the quality experiment above remains necessary.


In [ ]:
position_table = base_vit_state["encoder.pos_embedding"][:, 1:]  # exclude CLS position
source_side = int(math.sqrt(position_table.shape[1]))
assert source_side * source_side == position_table.shape[1] == 196
source_position_grid = position_table.reshape(1, source_side, source_side, -1).permute(0, 3, 1, 2)
target_side = 24
target_position_grid = F.interpolate(
    source_position_grid, size=(target_side, target_side), mode="bicubic", align_corners=True
)

spatial_variation = source_position_grid.flatten(2).std(dim=-1).squeeze(0)
position_dimensions = spatial_variation.topk(3).indices.tolist()
position_interpolation_summary = pd.DataFrame([
    {
        "embedding_dimension": int(dimension),
        "source_grid": f"{source_side}×{source_side}",
        "target_grid": f"{target_side}×{target_side}",
        "source_std": float(source_position_grid[0, dimension].std()),
        "target_std": float(target_position_grid[0, dimension].std()),
    }
    for dimension in position_dimensions
])
display(position_interpolation_summary.round(5))

fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
for column, dimension in enumerate(position_dimensions):
    source_map = source_position_grid[0, dimension].numpy()
    target_map = target_position_grid[0, dimension].numpy()
    lower, upper = min(source_map.min(), target_map.min()), max(source_map.max(), target_map.max())
    axes[0, column].imshow(source_map, cmap="coolwarm", vmin=lower, vmax=upper)
    axes[0, column].set_title(f"Learned 14×14 · dim {dimension}")
    axes[1, column].imshow(target_map, cmap="coolwarm", vmin=lower, vmax=upper)
    axes[1, column].set_title(f"Interpolated 24×24 · dim {dimension}")
    axes[0, column].axis("off"); axes[1, column].axis("off")
plt.suptitle("Absolute position embeddings: same dimensions, resized spatial grid")
plt.tight_layout()


## Phase 8 — Inspect official ViT attention safely

The function below mirrors the official torchvision encoder path while requesting per-head weights. It first asserts that the reproduced final representation matches `vit_encoder(images)`. Only then are the weights used as an interaction diagnostic.

**Important:** raw attention is not a causal explanation. Residual connections, values, MLPs, later layers, and the logistic probe all affect the final prediction.


In [ ]:
def vit_forward_with_attention(model: nn.Module, images: torch.Tensor, capture_layers=(0, 5, 11)):
    model.eval()
    x = model._process_input(images)
    cls = model.class_token.expand(images.shape[0], -1, -1)
    x = torch.cat([cls, x], dim=1)
    x = model.encoder.dropout(x + model.encoder.pos_embedding)
    captured = {}
    for index, layer in enumerate(model.encoder.layers):
        normalized = layer.ln_1(x)
        attended, weights = layer.self_attention(
            normalized, normalized, normalized, need_weights=True, average_attn_weights=False
        )
        x = x + layer.dropout(attended)
        x = x + layer.mlp(layer.ln_2(x))
        if index in capture_layers:
            captured[index] = weights.detach().cpu()
    representation = model.encoder.ln(x)[:, 0]
    return representation, captured


vit_encoder = vit_encoder.to("cpu").eval()
attention_transform = transform_at(224)
attention_batch = torch.stack([attention_transform(row.image) for row in test_rows])
with torch.inference_mode():
    official_representation = vit_encoder(attention_batch)
    reproduced_representation, captured_attention = vit_forward_with_attention(vit_encoder, attention_batch)
max_representation_error = (official_representation - reproduced_representation).abs().max().item()
assert torch.allclose(official_representation, reproduced_representation, atol=1e-5, rtol=1e-4)
print("official path equivalence max error:", max_representation_error)
print({layer: tuple(value.shape) for layer, value in captured_attention.items()})


### Attention gallery: correct, hard/incorrect, small, large, and shifted

All examples are Factory C source-shift samples. If this tiny test split has no incorrect prediction, the notebook substitutes the lowest-confidence example and labels it honestly.


In [ ]:
truth = np.array([row.label for row in test_rows])
confidence = vit_test_probabilities.max(axis=1)
correct_indices = np.where(vit_test_predictions == truth)[0]
incorrect_indices = np.where(vit_test_predictions != truth)[0]
small_indices = np.where([row.small_defect for row in test_rows])[0]
large_indices = np.where([(row.label != 0 and not row.small_defect) for row in test_rows])[0]
gallery = [
    ("correct", int(correct_indices[0]) if len(correct_indices) else int(np.argmax(confidence))),
    ("incorrect" if len(incorrect_indices) else "lowest confidence", int(incorrect_indices[0]) if len(incorrect_indices) else int(np.argmin(confidence))),
    ("small defect", int(small_indices[0])),
    ("large defect", int(large_indices[0])),
    ("source shift", 0),
]

last_attention = captured_attention[11].mean(dim=1)[:, 0, 1:].reshape(len(test_rows), 1, 14, 14)
maps = F.interpolate(last_attention, size=(224, 224), mode="bilinear", align_corners=False).squeeze(1).numpy()
fig, axes = plt.subplots(len(gallery), 2, figsize=(8, 3.1 * len(gallery)))
for row_index, (slice_name, index) in enumerate(gallery):
    image = np.asarray(test_rows[index].image)
    attention_map = maps[index]
    attention_map = (attention_map - attention_map.min()) / (np.ptp(attention_map) + 1e-8)
    axes[row_index, 0].imshow(image); axes[row_index, 0].axis("off")
    axes[row_index, 0].set_title(f"{slice_name}: true={CLASS_NAMES[truth[index]]}\npred={CLASS_NAMES[vit_test_predictions[index]]}, p={confidence[index]:.2f}")
    axes[row_index, 1].imshow(image); axes[row_index, 1].imshow(attention_map, cmap="magma", alpha=0.55)
    axes[row_index, 1].axis("off"); axes[row_index, 1].set_title("Last-layer mean CLS attention\ninteraction diagnostic—not explanation")
plt.tight_layout()


### One example, two attribution questions

For one small-defect image, compare last-layer class-token attention with gradient-times-input attribution for the frozen ViT representation plus its trained linear probe.

- **Attention** asks how one transformer layer mixed token representations.
- **Gradient attribution** asks how locally sensitive the selected probe score is to the current input.

Neither is causal ground truth. Gradients can be noisy or saturated; attention omits values, residual paths, MLPs, other layers, and the downstream classifier. The goal is to see that different methods answer different questions and can produce different pictures.


In [ ]:
attribution_index = int(small_indices[0])
predicted_label = int(vit_test_predictions[attribution_index])
scaler = vit_probe.named_steps["standardscaler"]
classifier = vit_probe.named_steps["logisticregression"]
class_row = int(np.where(classifier.classes_ == predicted_label)[0][0])
safe_scale = np.where(scaler.scale_ == 0, 1.0, scaler.scale_)
effective_probe_weight = torch.tensor(
    classifier.coef_[class_row] / safe_scale, dtype=attention_batch.dtype
)

gradient_input = attention_batch[attribution_index:attribution_index + 1].clone().requires_grad_(True)
vit_encoder.zero_grad(set_to_none=True)
representation = vit_encoder(gradient_input)
selected_score = (representation.squeeze(0) * effective_probe_weight).sum()
selected_score.backward()
gradient_times_input = (gradient_input.grad * gradient_input).abs().mean(dim=1).squeeze(0).detach().numpy()
gradient_times_input = (gradient_times_input - gradient_times_input.min()) / (np.ptp(gradient_times_input) + 1e-8)

attention_overlay = maps[attribution_index]
attention_overlay = (attention_overlay - attention_overlay.min()) / (np.ptp(attention_overlay) + 1e-8)
source_image = np.asarray(test_rows[attribution_index].image)
attribution_comparison = {
    "sample_id": test_rows[attribution_index].sample_id,
    "true_label": CLASS_NAMES[test_rows[attribution_index].label],
    "predicted_label": CLASS_NAMES[predicted_label],
    "attention_method": "last-layer mean CLS-to-patch attention",
    "gradient_method": "absolute gradient-times-input for standardized linear-probe score",
    "interpretation": "complementary diagnostics; neither is causal ground truth",
}
print(attribution_comparison)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(source_image); axes[0].set_title("Input")
axes[1].imshow(source_image); axes[1].imshow(attention_overlay, cmap="magma", alpha=0.55)
axes[1].set_title("Attention\ninternal token mixing")
axes[2].imshow(source_image); axes[2].imshow(gradient_times_input, cmap="viridis", alpha=0.55)
axes[2].set_title("Gradient × input\nlocal score sensitivity")
for axis in axes:
    axis.axis("off")
plt.suptitle("Different methods, different questions—not competing ground truths")
plt.tight_layout()


### Attention distance across depth

For every patch query, expected distance is the attention-weighted spatial distance to patch keys. The calculation excludes the class token and normalizes by the patch-grid diagonal. Head variation is preserved instead of collapsing everything to one heatmap.


In [ ]:
grid = 14
coordinates = torch.stack(torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij"), dim=-1).reshape(-1, 2).float()
distance = torch.cdist(coordinates, coordinates) / math.sqrt(2 * (grid - 1) ** 2)
distance_records = []
for layer_index, attention in captured_attention.items():
    patch_attention = attention[:, :, 1:, 1:]
    expected = (patch_attention * distance).sum(dim=-1).mean(dim=-1)  # batch × heads
    for head_index in range(expected.shape[1]):
        values = expected[:, head_index].numpy()
        distance_records.append({
            "layer": layer_index + 1,
            "head": head_index,
            "mean_normalized_distance": float(values.mean()),
            "std_across_images": float(values.std()),
        })

attention_distance = pd.DataFrame(distance_records)
display(attention_distance.groupby("layer").mean(numeric_only=True).round(4))
fig, axis = plt.subplots(figsize=(9, 4))
for layer_index, group in attention_distance.groupby("layer"):
    axis.scatter([layer_index] * len(group), group.mean_normalized_distance, alpha=0.75, label=f"Layer {layer_index}")
axis.set(xlabel="Encoder layer", ylabel="Mean normalized attention distance", title="Heads differ; distance does not imply causal importance")
axis.set_xticks(sorted(attention_distance.layer.unique())); plt.tight_layout()


### Token-level representation evolution

Attention weights describe mixing, but the representation itself is what later blocks and the probe consume. Track one image from patch-plus-position tokens through blocks 1, 4, 8, and 12. Four descriptive signals make contextualization visible:

- class-token-to-patch cosine similarity;
- average patch-to-patch cosine similarity;
- spatial-neighbor similarity; and
- variance across patch tokens.

No direction is inherently good. Rising similarity can mean useful integration or collapse; falling similarity can mean specialization or instability. Interpret it with task metrics and failure slices.


In [ ]:
def collect_token_states(model: nn.Module, image: torch.Tensor, capture=(1, 4, 8, 12)) -> dict[str, torch.Tensor]:
    model.eval()
    with torch.inference_mode():
        tokens = model._process_input(image)
        cls = model.class_token.expand(image.shape[0], -1, -1)
        tokens = model.encoder.dropout(torch.cat([cls, tokens], dim=1) + model.encoder.pos_embedding)
        states = {"Patch + position": tokens.detach().cpu()}
        for layer_number, layer in enumerate(model.encoder.layers, start=1):
            tokens = layer(tokens)
            if layer_number in capture:
                states[f"Block {layer_number}"] = tokens.detach().cpu()
    return states


evolution_image = attention_batch[attribution_index:attribution_index + 1]
token_states = collect_token_states(vit_encoder, evolution_image)
token_evolution_records = []
token_similarity_matrices = {}
for stage_index, (stage, state) in enumerate(token_states.items()):
    normalized = F.normalize(state.squeeze(0), dim=-1)
    cls_token, patches = normalized[0], normalized[1:]
    pairwise = patches @ patches.T
    off_diagonal = pairwise[~torch.eye(pairwise.shape[0], dtype=torch.bool)]
    patch_grid = patches.reshape(14, 14, -1)
    horizontal = (patch_grid[:, :-1] * patch_grid[:, 1:]).sum(dim=-1)
    vertical = (patch_grid[:-1, :] * patch_grid[1:, :]).sum(dim=-1)
    token_evolution_records.append({
        "stage_index": stage_index,
        "stage": stage,
        "cls_to_patch_similarity": float((patches @ cls_token).mean()),
        "average_patch_similarity": float(off_diagonal.mean()),
        "spatial_neighbor_similarity": float(torch.cat([horizontal.flatten(), vertical.flatten()]).mean()),
        "token_variance": float(state.squeeze(0)[1:].var(dim=0).mean()),
    })
    token_similarity_matrices[stage] = pairwise.numpy()

token_evolution = pd.DataFrame(token_evolution_records)
display(token_evolution.round(5))

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for axis, metric, title in [
    (axes[0, 0], "cls_to_patch_similarity", "CLS ↔ patch similarity"),
    (axes[0, 1], "average_patch_similarity", "Average patch similarity"),
    (axes[1, 0], "spatial_neighbor_similarity", "Spatial-neighbor similarity"),
    (axes[1, 1], "token_variance", "Token variance"),
]:
    axis.plot(token_evolution.stage_index, token_evolution[metric], marker="o")
    axis.set_xticks(token_evolution.stage_index, token_evolution.stage, rotation=25, ha="right")
    axis.set_title(title)
plt.suptitle("How one image's token representation changes through depth")
plt.tight_layout()

fig, axes = plt.subplots(1, len(token_similarity_matrices), figsize=(17, 3.5))
for axis, (stage, matrix) in zip(axes, token_similarity_matrices.items()):
    axis.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
    axis.set_title(stage)
    axis.set_xlabel("Patch token"); axis.set_ylabel("Patch token")
plt.suptitle("Patch-token cosine-similarity matrices")
plt.tight_layout()


## Phase 9 — Record the evidence and make a provisional decision

The selection below ranks held-out-source macro F1, then small-defect recall, then measured batch-1 median latency. That rule is explicit so it can be challenged. A real release gate would use minimum quality thresholds, target-device p95 latency, export parity, calibration, cost, and human-review capacity.


In [ ]:
ranked = merged_results.sort_values(
    ["macro_f1", "small_defect_recall", "median_ms_b1"], ascending=[False, False, True]
).reset_index(drop=True)
selected = ranked.iloc[0]
CONTRACT_THRESHOLD_NOTICE = "Demonstration thresholds and measurements for this notebook runtime only; validate the exported model on the exact target device."

def records_without_nan(frame: pd.DataFrame):
    return json.loads(frame.to_json(orient="records"))


decision = {
    "course": "Beginner 03 — Vision Transformers",
    "status": "provisional teaching decision, not production approval",
    "selection_rule": "highest held-out-source macro F1, then small-defect recall, then lower notebook median batch-1 latency",
    "selected_candidate": selected.model,
    "contract_notice": CONTRACT_THRESHOLD_NOTICE,
    "environment": {
        "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
        "device": str(DEVICE), "precision": "float32", "common_resolution": 224,
    },
    "fair_comparison_controls": [
        "same generated dataset and source-aware split", "same 224x224 tensor and ImageNet normalization",
        "frozen official torchvision weights", "same standardized logistic-regression probe family",
    ],
    "remaining_confounders": [
        "different upstream data and training recipes", "different capacities and representation widths",
        "generated data is not a factory acceptance dataset", "notebook runtime is not target hardware",
    ],
    "quality_results": records_without_nan(quality_results),
    "systems_results": records_without_nan(systems_results),
    "patch_size_experiment": records_without_nan(patch_results),
    "resolution_experiment": records_without_nan(resolution_results),
    "attention_distance": records_without_nan(attention_distance),
    "attention_memory": records_without_nan(attention_memory),
    "global_vs_window_cost": records_without_nan(attention_cost),
    "position_interpolation": records_without_nan(position_interpolation_summary),
    "attribution_comparison": attribution_comparison,
    "token_evolution": records_without_nan(token_evolution),
    "confusion_matrices": confusion_records,
    "required_next_gates": [
        "evaluate real source- and time-separated data", "validate calibration and abstention",
        "export through the intended runtime and verify parity", "remeasure p95 latency, memory, power, and throughput on target hardware",
        "review upstream license, provenance, and domain risks", "define monitoring, human review, rollback, and retention",
    ],
}

quality_results.to_csv(ARTIFACT_DIR / "frozen_probe_quality.csv", index=False)
systems_results.to_csv(ARTIFACT_DIR / "systems_profile.csv", index=False)
patch_results.to_csv(ARTIFACT_DIR / "patch_size_experiment.csv", index=False)
resolution_results.to_csv(ARTIFACT_DIR / "resolution_shift.csv", index=False)
attention_memory.to_csv(ARTIFACT_DIR / "attention_memory.csv", index=False)
attention_cost.to_csv(ARTIFACT_DIR / "global_vs_window_cost.csv", index=False)
position_interpolation_summary.to_csv(ARTIFACT_DIR / "position_interpolation.csv", index=False)
token_evolution.to_csv(ARTIFACT_DIR / "token_evolution.csv", index=False)
(ARTIFACT_DIR / "transformer_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps({"selected": selected.model, "notice": CONTRACT_THRESHOLD_NOTICE}, indent=2))
print("saved:", ARTIFACT_DIR / "transformer_decision.json")


## What you should now be able to explain without code

1. Why does a 224-pixel image create 196 patch tokens at patch size 16?
2. Why does halving patch size increase global-attention score entries by roughly 16×?
3. Why is non-overlapping patch projection equivalent to a strided convolution?
4. What ambiguity remains if identical patch tokens have no position information?
5. What distinct jobs do queries, keys, and values perform?
6. Why is the $\sqrt{d_k}$ scale factor needed?
7. Why can a CNN still represent long-range relationships?
8. Why did data scale and recipe matter for early ViTs?
9. How do shifted windows move information across Swin stages?
10. Why is raw attention not automatically a faithful explanation?
11. Why does position interpolation repair shapes without proving resolution robustness?
12. Why is the highest-quality model not automatically the best deployment choice?

## Extension exercises

- Compare class-token and mean-pooling tiny ViTs under identical seeds.
- Inject a patch-order permutation with and without position embeddings.
- Add attention rollout and compare it with raw last-layer attention and occlusion tests.
- Export one candidate and re-run the timing protocol in ONNX Runtime or the device runtime you actually intend to use.
- Replace the generated data with a licensed, source-aware inspection dataset and document the new evidence boundary.

## Transition to Course 04 — where do representations come from?

ViTs work extremely well with large-scale pretraining. But architecture only determines how information **can** flow. The next question is what learning signal shapes the representation:

```text
Supervised labels
      ↓
task-defined semantic categories

Contrastive learning
      ↓
relationships between augmented examples

Masked image modeling
      ↓
predict missing visual information

Teacher–student learning
      ↓
learn stable invariances without manual labels
```

> Architecture determines how information can flow. Pretraining determines what representations it learns.

Course 04 studies these objectives, collapse prevention, frozen probing, and dense versus global features. It will teach DINO/MAE-family mechanics there rather than expanding Course 03 with more transformer families.
